In [ ]:
import os
import json
import gc
import torch
from tqdm import tqdm
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# ==========================================
# 1. 核心路径配置（完美锁定你的真实路径）
# ==========================================
ANNOTATION_PATH = r"G:\sample\Qwen3-VL\AMBER-master\data\query\query_all.json"
IMAGE_DIR = r"G:\sample\Qwen3-VL\AMBER\image" 
OUTPUT_PATH = r"G:\sample\Qwen3-VL\AMBER-master\qwen3_baseline_output_new.json"

BATCH_SAVE_SIZE = 20  # 每 20 步定时存盘

# ==========================================
# 2. 联网加载/补齐完整的 Qwen3-VL-2B-Instruct
# ==========================================
print("正在加载 Qwen3-VL 处理器与模型...")
model_id = "Qwen/Qwen3-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,  # 必须使用 bf16 格式保护 5060 显存
    device_map="cuda",
    low_cpu_mem_usage=True
).eval()
# ====================================================================
# 【新插一格】动态计算并注入 VTI 干预层（完美适配原生 add_vti_layers 接口）
# ====================================================================
def load_and_resize_images(image_dir, max_pixels=512*512, max_count=5):
    images = []
    for f in os.listdir(image_dir)[:max_count]:
        if not f.lower().endswith(('.jpg','.png','.jpeg')): continue
        img_path = os.path.join(image_dir, f)
        img = Image.open(img_path).convert("RGB")
        # 限制像素总数不超过 max_pixels
        w, h = img.size
        if w * h > max_pixels:
            ratio = (max_pixels / (w * h)) ** 0.5
            new_size = (int(w * ratio), int(h * ratio))
            img = img.resize(new_size, Image.Resampling.LANCZOS)
        images.append(img)
    return images

demo_images = load_and_resize_images(IMAGE_DIR, max_pixels=256*256, max_count=5)
import os
import json
import gc
import torch
from tqdm import tqdm
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import torch
from compute_vti import compute_visual_direction
from vti_core import add_vti_layers, remove_vti_layers

# 1. 抽取当前数据集前 5 张图作为提取“幻觉方向向量 d”的样本源
# 完美匹配你的 run_amber_baseline.ipynb 中的变量名 target_data
from PIL import Image

image_dir = r"G:\sample\Qwen3-VL\AMBER\image"

demo_images = []

# 读取前2张图片
for file in os.listdir(image_dir)[:2]:

    if file.lower().endswith((
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp"
    )):

        img_path = os.path.join(image_dir, file)

        try:
            img = Image.open(img_path).convert("RGB")
            demo_images.append(img)

        except Exception as e:
            print(f"读取失败: {img_path}")
            print(e)

print(f"✅ 成功加载 {len(demo_images)} 张图片")
print("⏳ 正在抽取视觉特征空间并进行 PCA 降维，请稍候...")
# 运行计算，得出对抗幻觉的特征纠正方向
visual_dir = compute_visual_direction(model, processor, demo_images, rank=1,mask_ratio=0.9,num_trials=3)
print(f"🎯 幻觉干涉方向向量计算完成！维度: {visual_dir.shape}")

# 2. 靶向注入：调用你原生的 add_vti_layers 函数挂载到 Qwen3-VL
# alpha 设为 0.5 开启干预强度
model = add_vti_layers(model, directions=visual_dir, alpha=0.5)
print("✅ VTI 干预层已成功嵌入模型前向传播！")

# ==========================================
# 3. 读取并严格筛选前 500 个生成式样本
# ==========================================
with open(ANNOTATION_PATH, "r", encoding="utf-8") as f:
    amber_data = json.load(f)

# 【核心截断】：只拿前 500 个样本进行生成式评测
target_data = amber_data[:500]
print(f"--- 核心调试：成功锁定前 {len(target_data)} 个生成式测试样本！ ---")

# 断点续训去重机制
results = {}
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        results = json.load(f)
    print(f"检测到历史进度，已跳过前 {len(results)} 条已处理的题目。")

# ==========================================
# 4. 生成式高速推理循环
# ==========================================
print("开始在 AMBER 前 500 个生成式样本上进行推理...")

for idx, item in enumerate(tqdm(target_data)):
    img_id = str(item.get("id", idx))
    
    # 如果已经跑过了，直接跳过（方便中途断线恢复）
    if img_id in results:
        continue
        
    img_name = item.get("image", f"{img_id}.jpg")
    img_path = os.path.join(IMAGE_DIR, img_name)
    
    # 路径兜底兼容
    if not os.path.exists(img_path):
        if not img_name.endswith(".jpg"):
            img_path = os.path.join(IMAGE_DIR, img_name + ".jpg")
        if not os.path.exists(img_path):
            print(f"警告：找不到图片 {img_path}，跳过。")
            continue

    # 提取生成式 Prompt (通常为 Describe this image.)
    prompt = item.get("query", "Describe this image.")
    
    try:
        image = Image.open(img_path)
        # 【控显存核心】：压到 384 分辨率，防止生成长文本时 KV Cache 撑爆 8GB 显存
        image.thumbnail((384, 384), Image.Resampling.LANCZOS)
        image = image.convert("RGB")
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        )
        
        inputs.pop("token_type_ids", None)
        inputs = {k: v.to("cuda") for k, v in inputs.items() if isinstance(v, torch.Tensor)}

        with torch.inference_mode():
            # 生成式任务：放宽到 64 token 保证模型能完整描述图片
            generated_ids = model.generate(**inputs, max_new_tokens=64)
            generated_ids_trimmed = [
                out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
            ]
            output_text = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0].strip()

        results[img_id] = output_text

    except Exception as e:
        print(f"处理题目 ID {img_id} 时发生错误: {e}")
        continue

    # ==========================================
    # 5. 主动定时存盘与显存清理
    # ==========================================
    if (idx + 1) % BATCH_SAVE_SIZE == 0:
        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=4)
        
        if 'inputs' in locals(): del inputs
        if 'generated_ids' in locals(): del generated_ids
        if 'image' in locals(): del image
        gc.collect()
        torch.cuda.empty_cache()

# 最终全量保存
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"完成结果存至: {OUTPUT_PATH}")

c:\Users\Lenovo\.conda\envs\qwen3_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在加载 Qwen3-VL 处理器与模型...


Loading weights: 100%|██████████| 625/625 [00:01<00:00, 493.28it/s]


✅ 成功加载 5 张图片
⏳ 正在抽取视觉特征空间并进行 PCA 降维，请稍候...


Extracting visual states:  60%|██████    | 3/5 [04:16<02:51, 85.62s/it]


KeyboardInterrupt: 